# Air Quality ML Analysis — Trust, Events & Forecasts

**Data:** [DrKoz/ACS_AQ](https://github.com/DrKoz/ACS_AQ) · 5 sensors · Nov 2025 · 2-min resolution

We build three components that feed the monitoring dashboard:

1. **Trust scores** — from dual-sensor A/B agreement (how much can we rely on each reading?)
2. **Event detection** — smoke spikes, rain washout, sensor failures
3. **Short-term forecast** — PM2.5 prediction with calibrated uncertainty intervals

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from load_data import load_all_sensors
from ml_pipeline import (
    add_trust_scores,
    compute_trust_score,
    detect_events,
    run_full_pipeline,
    train_forecast_with_uncertainty,
    FULL_SENSORS,
)

---
## 1. Trust Score — Dual-Sensor Agreement

Each PurpleAir-style unit has two channels (A and B). When they agree, we trust the reading more. We compute:

$$\text{trust} = 100 \cdot e^{-3 \cdot |A-B|/\bar{x}}$$

High agreement → trust near 100. Large divergence → trust drops toward 0.

In [ ]:
df = load_all_sensors()
df = add_trust_scores(df)

trust_by_sensor = df.groupby("sensor_index")["trust_score"].agg(["mean", "std", "count"]).round(2)
trust_by_sensor

In [ ]:
# Visual style: dark, technical, no chartjunk
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.facecolor": "#0d0d0d",
    "figure.facecolor": "#0a0a0a",
    "axes.edgecolor": "#2a2a2a",
    "axes.labelcolor": "#b0b0b0",
    "xtick.color": "#888",
    "ytick.color": "#888",
    "grid.color": "#1a1a1a",
    "grid.alpha": 0.8,
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor="#0a0a0a")

# A: Trust distribution by sensor
ax = axes[0]
sensors = [230019, 249443, 249445, 290694, 297697]
colors = ["#4fc3f7", "#81c784", "#ffb74d", "#e57373", "#ba68c8"]
for i, sid in enumerate(sensors):
    vals = df[df["sensor_index"] == sid]["trust_score"].dropna()
    if len(vals) > 0:
        ax.hist(vals, bins=50, alpha=0.6, color=colors[i], label=str(sid), edgecolor="none")
ax.set_xlabel("Trust score")
ax.set_ylabel("Count")
ax.set_title("Trust distribution by sensor")
ax.legend(loc="upper left", fontsize=8)
ax.set_xlim(0, 105)
ax.grid(True, alpha=0.3)

# B: Trust over time (one sensor)
ax = axes[1]
s = df[df["sensor_index"] == 249443].copy()
s = s.sort_values("time_stamp").tail(2000)
ax.fill_between(s["time_stamp"], s["trust_score"], alpha=0.4, color="#4fc3f7")
ax.plot(s["time_stamp"], s["trust_score"], color="#4fc3f7", linewidth=0.8)
ax.set_xlabel("Time")
ax.set_ylabel("Trust score")
ax.set_title("Trust over time — Sensor 249443")
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("fig_trust.png", dpi=150, facecolor="#0a0a0a", edgecolor="none", bbox_inches="tight")
plt.show()
print("Saved fig_trust.png")

---
## 2. Event Detection — Smoke, Washout, Sensor Failure

Rule-based detection:
- **Smoke spike:** Rapid PM2.5 rise (>50% in 30 min) and sustained high (>35 µg/m³)
- **Rain washout:** Rapid PM2.5 drop (>35% in 1 h) — precipitation clears particles
- **Sensor failure:** A/B divergence (trust < 40%) — calibration drift

In [ ]:
out = run_full_pipeline()
events = out["events"]
print(f"Detected {len(events)} events in last 50")
for e in events[:8]:
    print(f"  {e['type']:14} | {e['sensor']} | {e['time']} | {e['severity']} | {e['desc'][:50]}...")

In [ ]:
event_df = pd.DataFrame(events)
if len(event_df) > 0:
    type_counts = event_df["type"].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4), facecolor="#0a0a0a")
    colors = {"smoke_spike": "#ff6b35", "rain_washout": "#4fc3f7", "sensor_failure": "#ffd54f"}
    bars = ax.bar(range(len(type_counts)), type_counts.values, color=[colors.get(t, "#666") for t in type_counts.index], edgecolor="none")
    ax.set_xticks(range(len(type_counts)))
    ax.set_xticklabels([t.replace("_", " ").title() for t in type_counts.index], fontsize=10)
    ax.set_ylabel("Count")
    ax.set_title("Event types detected")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig_events.png", dpi=150, facecolor="#0a0a0a", edgecolor="none", bbox_inches="tight")
    plt.show()
    print("Saved fig_events.png")

---
## 3. Forecast with Uncertainty — 80% Prediction Intervals

Random Forest regressor with lag features (past 12 steps), temp, humidity, hour, day. We use **bootstrap over trees** to get 80% intervals: each tree predicts; we take 10th and 90th percentiles.

In [ ]:
results = []
for sid in FULL_SENSORS:
    if (df["sensor_index"] == sid).sum() < 5000:
        continue
    r = train_forecast_with_uncertainty(df, sensor_id=sid, n_lags=12, horizon=1, test_frac=0.2)
    results.append(r)

print("Forecast accuracy (next-step PM2.5):")
for r in results:
    print(f"  Sensor {r['sensor_id']}: MAE={r['mae']:.3f} µg/m³  R²={r['r2']:.3f}")

In [ ]:
best = max(results, key=lambda x: x["r2"])
y_test = best["y_test"]
y_pred = best["y_pred"]
y_lo = best["y_lo"]
y_hi = best["y_hi"]
times = best["times_test"]
sid = best["sensor_id"]

n_show = min(400, len(y_test))
t = times.iloc[-n_show:]
y_t = y_test.iloc[-n_show:]
y_p = y_pred[-n_show:]
y_l = y_lo.iloc[-n_show:]
y_h = y_hi.iloc[-n_show:]

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="#0a0a0a")
ax.fill_between(t, y_l, y_h, alpha=0.25, color="#4fc3f7")
ax.plot(t, y_t, color="#e0e0e0", linewidth=1.0, label="Actual")
ax.plot(t, y_p, color="#4fc3f7", linewidth=0.9, alpha=0.9, label="Forecast")
ax.set_xlabel("Time")
ax.set_ylabel("PM₂.₅ (µg/m³)")
ax.set_title(f"Sensor {sid}: Forecast vs actual (80% prediction interval)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("fig_forecast.png", dpi=150, facecolor="#0a0a0a", edgecolor="none", bbox_inches="tight")
plt.show()
print("Saved fig_forecast.png")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), facecolor="#0a0a0a")
ax.scatter(y_test, y_pred, alpha=0.35, s=6, color="#4fc3f7", edgecolors="none")
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
pad = (hi - lo) * 0.02 or 0.5
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#888", ls="--", lw=1.2, label="1:1")
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_xlabel("Actual PM₂.₅ (µg/m³)")
ax.set_ylabel("Predicted PM₂.₅ (µg/m³)")
ax.set_title(f"Forecast accuracy — Sensor {sid}")
ax.text(0.05, 0.95, f"R² = {best['r2']:.3f}\nMAE = {best['mae']:.3f} µg/m³", transform=ax.transAxes,
        fontsize=9, verticalalignment="top", color="#b0b0b0",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#111", edgecolor="#2a2a2a"))
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("fig_scatter.png", dpi=150, facecolor="#0a0a0a", edgecolor="none", bbox_inches="tight")
plt.show()
print("Saved fig_scatter.png")

---
## Summary

| Component | What we did | Output for dashboard |
|-----------|-------------|----------------------|
| **Trust** | Exponential decay from A/B divergence | `trust_score` per reading (0–100) |
| **Events** | Rule-based: spike (≥50% rise), washout (≥35% drop), A/B failure | Event list with type, time, severity |
| **Forecast** | RF + lag features; 80% interval from tree bootstrap | Point forecast + lo/hi bands |